# Individual Exercise: Mini Data Quality Audit

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
repo_root = Path.cwd().resolve().parents[1]
data_path = 'D:\\GERMANY\\SRH_MScADSA\\Sem2\\DM\\week2_customer_transactions_messy.csv'
df = pd.read_csv(data_path)
print('Loaded:', data_path)
print('Shape:', df.shape)
df.head()


Loaded: D:\GERMANY\SRH_MScADSA\Sem2\DM\week2_customer_transactions_messy.csv
Shape: (11, 9)


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09


## Task 1 - Dataset description

### Your answer

The dataset **week2_customer_transactions_messy.csv** contains **11 records** with **9 columns**:
- `transaction_id`: Unique identifier for each transaction
- `customer_id`: Customer identifier (some missing)
- `transaction_date`: Date of transaction (mixed formats)
- `amount`: Transaction amount in float
- `currency`: Currency code (EUR)
- `payment_method`: Payment type (credit_card, etc.)
- `status`: Transaction status (completed, pending, cancelled)
- `region`: Geographic region (inconsistent case)
- `last_updated`: Last update timestamp (some missing)

**Business Use Case**: This appears to be a **customer transaction log** for a retail/e-commerce business. It can be used for:
- Revenue analysis and forecasting
- Customer behavior analytics
- Payment method performance tracking
- Regional sales performance


## Task 2 - Issues by dimension

### Identified Data Quality Issues

| Issue | Dimension | Impact |
|-------|-----------|--------|
| Missing customer_id (row 4) | Completeness | Impacts customer analytics and tracking |
| Duplicate transaction_id 'T0006' (rows 6-7) | Uniqueness | May double count revenue |
| Inconsistent date formats (2026-01-05, 2026/01/06, 06-01-2026) | Validity | Parsing errors, analysis issues |
| Invalid date '2026-02-30' (row 8) | Validity | Data entry error, not a real date |
| Inconsistent region case ('DE' vs 'de') | Consistency | Grouping/aggregation problems |
| Missing last_updated (row 9) | Completeness | Audit trail incomplete |


In [8]:
issue_table = pd.DataFrame([['Missing customer_id','Completeness','Impacts customer analytics'],['Duplicate transaction_id','Uniqueness','May double count revenue']], columns=['Issue','Dimension','Impact'])
issue_table


,Issue,Dimension,Impact
0,Missing customer_id,Completeness,Impacts customer analytics
1,Duplicate transaction_id,Uniqueness,May double count revenue


## Task 3 - KPI calculations


In [9]:
kpis={}
kpis['Completeness Rate']=1-(df.isna().sum().sum()/(df.shape[0]*df.shape[1]))
kpis['Duplication Rate']=df.duplicated(subset=['transaction_id']).mean()
amount=pd.to_numeric(df['amount'], errors='coerce')
date_ok=pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').notna()
kpis['Error Rate']=(amount.isna() | (amount<0) | ~date_ok).mean()
pd.DataFrame({'KPI':list(kpis.keys()), 'Value':list(kpis.values())})


,KPI,Value
0,Completeness Rate,0.959596
1,Duplication Rate,0.090909
2,Error Rate,0.272727


### Your KPI interpretation

**Interpretation of KPIs:**

1. **Completeness Rate (96.0%)**: Indicates that ~4% of data is missing (customer_id and last_updated fields). This is relatively good but still needs attention.

2. **Duplication Rate (9.1%)**: One duplicate transaction_id 'T0006' exists. This is moderate but can lead to double-counting revenue.

3. **Error Rate (27.3%)**: High error rate due to invalid dates (e.g., 2026-02-30) and potentially negative amounts. This significantly affects data integrity for time-series analysis.

**Strongest Signal**: The **Error Rate (27.3%)** is the strongest signal as it affects over a quarter of records, indicating systemic issues with data validation at entry.


## Task 4 - Validation rules


In [10]:
rules={
'transaction_id_required': df['transaction_id'].isna() | (df['transaction_id'].astype(str).str.strip()==''),
'amount_non_negative': pd.to_numeric(df['amount'], errors='coerce')<0,
'transaction_date_valid': pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').isna(),
}
pd.DataFrame({k:int(v.sum()) for k,v in rules.items()}, index=['affected_rows']).T


,affected_rows
transaction_id_required,0
amount_non_negative,1
transaction_date_valid,1


## Task 5 - Audit summary

### Summary Table

| Issue Type | Affected Rows | Severity | Recommended Next Action |
|------------|---------------|----------|------------------------|
| Duplicate transaction_id (T0006) | 1 | **High** | Remove duplicates, keep first occurrence |
| Invalid transaction_date (2026-02-30) | 1 | **High** | Correct to valid date (2026-02-28 or 2026-03-02) |
| Negative amount | 1 | Medium | Validate amount is non-negative |
| Missing customer_id | 1 | Medium | Impute from lookup table or flag for review |
| Missing last_updated | 1 | Low | Populate with current timestamp |
| Inconsistent region case (de vs DE) | 1 | Low | Standardize to uppercase |


In [11]:
audit_summary = pd.DataFrame([['Example issue',0,'Medium','Example next action']], columns=['issue_type','affected_rows','severity','recommended_next_action'])
audit_summary


,issue_type,affected_rows,severity,recommended_next_action
0,Example issue,0,Medium,Example next action


## Task 6 - Recommended cleaning steps for next chapter

- **Recommendation 1**: Remove duplicate transaction_ids - keep first occurrence, investigate root cause
- **Recommendation 2**: Standardize date formats - convert all dates to ISO format (YYYY-MM-DD) using pd.to_datetime with format='mixed'
- **Recommendation 3**: Impute missing customer_ids - either from transaction logs or mark as 'UNKNOWN' with flag
- **Recommendation 4**: Validate date ranges - ensure transaction_date is not in future and is a valid calendar date
- **Recommendation 5**: Standardize categorical fields - uppercase region codes, validate payment_method values
- **Recommendation 6**: Populate missing last_updated - use current timestamp for records without audit trail


## Reflection questions

1. **Which KPI gave the strongest signal?**
   The **Error Rate (27.3%)** gave the strongest signal because it affects over a quarter of all records, indicating systemic issues with data validation at entry.

2. **Which issue should be escalated first?**
   The **invalid transaction_date (2026-02-30)** should be escalated first as it represents a data integrity issue that could break date-based analysis and reporting.

3. **What additional metadata would improve this audit?**
   - Data source/timestamp for when data was extracted
   - User/system who created/modified records
   - Business rules documentation
   - Expected value ranges for amount (min/max)
   - Valid payment_method and status values
